In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("tickets_database.db")
cursor = conn.cursor()

cursor.execute("SELECT Ticket_ID, Ticket_Description, Created_On FROM tickets")
tickets = cursor.fetchall()

df = pd.DataFrame(tickets, columns=["Ticket_ID", "Ticket_Description", "Created_On"])
conn.close()

print(df.head())


   Ticket_ID                                 Ticket_Description  \
0          1  I'm having an issue with the {product_purchase...   
1          2  I'm having an issue with the {product_purchase...   
2          3  I'm facing a problem with my {product_purchase...   
3          4  I'm having an issue with the {product_purchase...   
4          5  I'm having an issue with the {product_purchase...   

         Created_On  
0  01-06-2023 12:15  
1  01-06-2023 16:45  
2  01-06-2023 11:14  
3  01-06-2023 07:29  
4  01-06-2023 00:12  


In [2]:
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
import torch
#variant of Facebook’s BART model fine-tuned for natural language inference (NLI)
model_name = "facebook/bart-large-mnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)  # 3 classes

labels = ["Low", "Medium", "High"]

classifier = pipeline("zero-shot-classification", model=model, tokenizer=tokenizer)

def predict_priority(description):
    result = classifier(description, candidate_labels=labels)
    priority = result["labels"][0]  # Highest probability label
    probabilities = {label: prob for label, prob in zip(result["labels"], result["scores"])}
    return priority, probabilities

sample_text = "Server is down, urgent critical issue!"
priority, probs = predict_priority(sample_text)
print(f"Priority: {priority}, Probabilities: {probs}")


c:\Users\Ritanya\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cpu


Priority: High, Probabilities: {'High': 0.6103360056877136, 'Low': 0.3134007155895233, 'Medium': 0.07626330852508545}


In [3]:

conn = sqlite3.connect("tickets_database.db")
cursor = conn.cursor()

cursor.execute("ALTER TABLE tickets ADD COLUMN priority TEXT")
cursor.execute("ALTER TABLE tickets ADD COLUMN priority_low FLOAT")
cursor.execute("ALTER TABLE tickets ADD COLUMN priority_medium FLOAT")
cursor.execute("ALTER TABLE tickets ADD COLUMN priority_high FLOAT")
conn.commit()


In [ ]:
import time

start_time = time.time()

for index, row in df.iterrows():
    print(f"Processing Ticket_ID: {row['Ticket_ID']}...")

    priority, probs = predict_priority(row["Ticket_Description"])

    print(f" → Predicted: {priority}")

    cursor.execute("UPDATE tickets SET priority=?, priority_low=?, priority_medium=?, priority_high=? WHERE Ticket_ID=?",
                   (priority, probs["Low"], probs["Medium"], probs["High"], row["Ticket_ID"]))

    if index % 10 == 0:
        conn.commit()
        print(f" ✅ Committed at Ticket_ID: {row['Ticket_ID']}")

conn.commit()
conn.close()

end_time = time.time()
print(f"✅ Priority predictions stored successfully! Total time: {end_time - start_time:.2f} sec")


Processing Ticket_ID: 1...
 → Predicted: High
 ✅ Committed at Ticket_ID: 1
Processing Ticket_ID: 2...
 → Predicted: High
Processing Ticket_ID: 3...
 → Predicted: High
Processing Ticket_ID: 4...
 → Predicted: High
Processing Ticket_ID: 5...
 → Predicted: High
Processing Ticket_ID: 6...
 → Predicted: High
Processing Ticket_ID: 7...
 → Predicted: High
Processing Ticket_ID: 8...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


 → Predicted: High
Processing Ticket_ID: 9...
 → Predicted: High
Processing Ticket_ID: 10...
 → Predicted: High
Processing Ticket_ID: 11...
 → Predicted: High
 ✅ Committed at Ticket_ID: 11
Processing Ticket_ID: 12...
 → Predicted: High
Processing Ticket_ID: 13...
 → Predicted: High
Processing Ticket_ID: 14...
 → Predicted: High
Processing Ticket_ID: 15...
 → Predicted: High
Processing Ticket_ID: 16...
 → Predicted: High
Processing Ticket_ID: 17...
 → Predicted: High
Processing Ticket_ID: 18...
 → Predicted: Low
Processing Ticket_ID: 19...
 → Predicted: Low
Processing Ticket_ID: 20...
 → Predicted: High
Processing Ticket_ID: 21...
 → Predicted: High
 ✅ Committed at Ticket_ID: 21
Processing Ticket_ID: 22...
 → Predicted: High
Processing Ticket_ID: 23...
 → Predicted: High
Processing Ticket_ID: 24...
 → Predicted: High
Processing Ticket_ID: 25...
 → Predicted: High
Processing Ticket_ID: 26...
 → Predicted: High
Processing Ticket_ID: 27...
 → Predicted: High
Processing Ticket_ID: 28...
 → 